# Notebook 2 (phase 2) — LLM-as-judge: calibrate it or do not use it

Notebook 1 stopped where plain code stops. Code can tell that an answer cites an excerpt;
it cannot tell whether the excerpt says what the answer claims. On SQuAD 2.0, 13 questions
that have no answer were answered anyway, every one with a valid citation. On QASPER, a
right answer worded differently from the reference counts as wrong.

Reading is needed. A language model can read at scale, and it is tempting to stop there:
ask a model "is this answer good?" and average the scores. This notebook builds that
**judge** and then does the part that is usually skipped: it measures the judge, against a
person, before believing a single number from it.

Everything is imported from `src/` and run as is. **Live cells** run on the fixtures with
scripted models (no key, no cost, no database). **Archive cells** read the calibration set
versioned under `data/golden/judge_calibration/` and, once it exists, the calibration
report. On 2026-11-05 the 50 items are drawn and **the human labels are still to be
given**: `make eval-annotate`, about an hour. Until then this notebook shows the method
end to end on four toy answers, and the real numbers have an empty place waiting.

## Part 1 — The problem

Two properties of an answer need a reader:

- **groundedness**: is every claim of the answer stated in the excerpts the system was
  given? (An answer can be true and not grounded: the model knew it from elsewhere. For a
  system that must answer from documents, that is a failure.)
- **relevance**: does the answer address the question that was asked? (An answer can be
  grounded and irrelevant: a true sentence about something else.)

A **judge** is a model given a **rubric** (the instructions of a grader: what to check,
what to ignore, how to conclude) and an item to grade. It is cheap and tireless. It is also
a model: it can be wrong while sounding sure, and nothing in its output says when.

The trap is in how its quality is usually reported. Take 50 answers of which 45 are good,
and a judge that says "good" to all 50 without reading:

| | |
|---|---|
| agreement with a human | 45 / 50 = **90%** |
| bad answers it caught | **0 of 5** |

"The judge agrees with us 90% of the time" was true and the judge judged nothing. Measured
that way, an uncalibrated judge is a generator of reassuring numbers. Two things fix it:
a measure of agreement that discounts luck (**Cohen's kappa**, part 3.2), and a set of
items where both verdicts actually occur (part 3.3).

In [1]:
# --- Configuration: edit, then "Run All" ---
import os

from llm_testbench.config import find_repo_root

SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"
ROOT = find_repo_root()  # the notebook runs from notebooks/phase-2/

# --- Experiment 1 (part 3.2): two graders, ten items --------------------------------
# True = "yes". Change the judge's labels and watch agreement and kappa part ways.
TOY_HUMAN = [True, True, True, True, True, True, True, True, False, False]
TOY_JUDGE = [True, True, True, True, True, True, True, True, True, True]

# --- Experiment 2 (parts 3.3 and 3.4): your labels on the four toy answers -----------
# question id -> (supported by the excerpt?, relevant to the question?), "y" or "n".
HUMAN_LABELS = {
    "q-river": ("y", "y"),  # "The Loire rises in the Massif Central [1]."
    "q-length": ("n", "y"),  # "The Loire is 1,006 kilometres long [1]." (not in the excerpt)
    "q-nobel": ("y", "y"),  # "She received the Nobel Prize in Chemistry in 1911 [1]."
    "q-born": ("y", "n"),  # "Marie Curie shared the Nobel Prize in Physics in 1903 [1]."
}
SWAPPED_LABEL = ("n", "y")  # the Loire answer shown against the Marie Curie excerpt

# --- Experiment 3 (part 3.6): which changes the bias probes apply --------------------
PERTURBATIONS_TO_RUN = ["repeat", "padded", "without_citations"]

In [2]:
import json
import tempfile
from collections import Counter
from pathlib import Path

from llm_testbench.config import Settings
from llm_testbench.eval.agreement import cohen_kappa
from llm_testbench.eval.annotate import annotate
from llm_testbench.eval.bias import PERTURBATIONS, format_bias, probe
from llm_testbench.eval.calibration import (
    CALIBRATION_DIR,
    CalibrationReport,
    build_items,
    calibrate,
    format_calibration,
    materialize,
    read_items,
    read_labels,
    write_items,
)
from llm_testbench.eval.generation import load_generation, run_generation
from llm_testbench.eval.judge import PROMPTS, Judge, JudgeOutput, Rubric, render_material
from llm_testbench.eval.loaders.squad import SquadV2Loader
from llm_testbench.generation.answer import AnswerBuilder
from llm_testbench.llm import ModelRef
from llm_testbench.llm.client import LLMClient
from llm_testbench.llm.prompts import PromptRegistry
from llm_testbench.llm.testing import Step, anthropic_message, scripted_anthropic_provider

fixtures = Settings(
    data_dir=ROOT / "data" / "cache", fixtures_dir=ROOT / "tests" / "fixtures", sample=True
)
prompts = PromptRegistry.from_settings(fixtures)
workdir = Path(tempfile.mkdtemp(prefix="judge-notebook-"))
GENERATOR = ModelRef.parse("anthropic/claude-sonnet-5-5")
JUDGE = ModelRef.parse("openai/gpt-5.6-luna")


def scripted(backend: str, *replies: str) -> LLMClient:
    """A client whose `backend` answers the given texts, in order, then repeats the last:
    the real provider code on a scripted network (no key, no cost)."""
    client = LLMClient.from_settings(fixtures)
    provider, _ = scripted_anthropic_provider(
        [Step(json_body=anthropic_message(reply)) for reply in replies], backend=backend
    )
    client.register_provider(backend, provider)
    return client


def verdict(answer: str, reason: str) -> str:
    """A judge's raw reply, as JSON: the reasons, then the verdict."""
    return json.dumps({"justification": reason, "verdict": answer})

## Part 2 — The options

**Who grades.**

| Option | Cost | What you get | Verdict |
|---|---|---|---|
| A person reads every answer | hours per run, every run | the truth, as far as one person is consistent | does not scale; needed once, as the reference |
| A judge model, trusted as is | cents | numbers | numbers about the judge as much as about the system |
| **A judge model, measured against a person on a small set first** | cents + one hour, once per rubric version | numbers, and how far to believe them | **built here** |
| A third-party evaluator (Ragas faithfulness, DeepEval, promptfoo assertions) | cents | the same kind of judge, with someone else's prompt | to be measured on the same 50 items, next to ours (awaiting a decision) |

**How the judge is asked.** Each line is a decision of [ADR 012](../../docs/adr/012-judge-and-calibration.md):

| Choice | Taken | Rejected, and why |
|---|---|---|
| verdict | **yes or no** | a score from 1 to 5: no two graders read a "3" the same way, and 50 items cannot calibrate five levels |
| prompt | **one rubric per property** | one prompt for everything: a well-supported answer drags its relevance up |
| order of the output | **justification, then verdict** | verdict first: the model commits, then writes the justification that fits |
| judge model | **another provider than the generator** | the same family: a model grades its own family's answers more kindly (self-preference) |
| what the judge is shown | **the item and nothing else** | which system wrote the answer, or what verdict is expected |

And one rule written **before any label existed**: a judge is used on a rubric only if its
kappa with the human labels is at least 0.6 on that rubric. Below, the rubric is revised
and measured again (twice at most), then a stronger judge is tried; if nothing reaches 0.6,
the metric is reported as *not measured*. A threshold chosen after seeing the result is
not a threshold.

## Part 3 — Guided reading

### 3.1 One rubric, one question, one verdict

A rubric is a versioned prompt: [`prompts/judge_groundedness.md`](../../prompts/judge_groundedness.md)
and [`prompts/judge_relevance.md`](../../prompts/judge_relevance.md). Each says how to
grade in three steps, and, as important, **what does not count**: whether the answer is
true in the real world, its length, its style, its citation markers. Those lines are aimed
at known weaknesses of judges (part 3.6).

[`Judge.grade`](../../src/llm_testbench/eval/judge.py#L103) sends the rubric as the system prompt and the item as the user
message, and asks for a structured reply, [`JudgeOutput`](../../src/llm_testbench/eval/judge.py#L46): a `justification`,
then a `verdict` that can only be `"yes"` or `"no"`. The order of the fields is the order
of writing.

What the judge reads is built by [`render_material`](../../src/llm_testbench/eval/judge.py#L74), and it differs by rubric:
groundedness gets the question, the excerpts and the answer; **relevance does not get the
excerpts**. Relevance is a property of the question and the answer alone; show the excerpts
and a judge starts rewarding answers for being supported.

The result is a [`Verdict`](../../src/llm_testbench/eval/judge.py#L56): the label, the justification, the model that
answered, the version and fingerprint of the rubric, the cost. A verdict is only comparable
with verdicts given under the same rubric version.

The live cell grades one toy answer on both rubrics. The judge is scripted: its two replies
are written below.

In [3]:
squad = SquadV2Loader(fixtures).load()
example = next(e for e in squad.examples if e.example_id == "q-born")
answer = "Marie Curie shared the Nobel Prize in Physics in 1903 [1]."
evidence = example.evidence(squad.info.key)

for rubric in Rubric:
    print(f"===== what the judge reads for {rubric.value} =====")
    print(
        render_material(rubric, question=example.question, answer=answer, evidence=evidence), "\n"
    )

print("===== the rubric (system prompt), groundedness =====")
print(prompts.render(PROMPTS[Rubric.GROUNDEDNESS]).text, "\n")
print("===== the reply the judge must give (JSON schema) =====")
print(json.dumps(JudgeOutput.model_json_schema()["properties"], indent=1), "\n")

judge = Judge(
    client=scripted(
        "openai",
        verdict("yes", "The excerpt states the 1903 Physics prize shared with Pierre Curie."),
        verdict("no", "The question asks for a city of birth; the answer gives a prize."),
    ),
    prompts=prompts,
    model=JUDGE,
)
for rubric in Rubric:
    graded = await judge.grade(rubric, question=example.question, answer=answer, evidence=evidence)
    print(
        f"{rubric.value:<13} verdict={'yes' if graded.label else 'no':<3} "
        f"rubric v{graded.prompt_version} — {graded.justification}"
    )

===== what the judge reads for groundedness =====
Question:
In which city was Marie Curie born?

Excerpts:
[1] squad_v2 › Marie_Curie › gold passage
Marie Curie received the Nobel Prize in Physics in 1903, shared with Pierre Curie and Henri Becquerel. She received the Nobel Prize in Chemistry in 1911.

Answer to grade:
Marie Curie shared the Nobel Prize in Physics in 1903 [1]. 

===== what the judge reads for relevance =====
Question:
In which city was Marie Curie born?

Answer to grade:
Marie Curie shared the Nobel Prize in Physics in 1903 [1]. 

===== the rubric (system prompt), groundedness =====
You are grading an answer produced by a question-answering system. The system was given numbered excerpts and had to answer from them alone. Your only task: decide whether the answer is SUPPORTED by the excerpts.

How to grade:
1. Split the answer into its factual claims.
2. For each claim, look for a sentence of the excerpts that states it or directly entails it.
3. The verdict is "yes" on

That answer is the case that separates the two rubrics: every word of it is in the excerpt
(grounded: yes), and it does not answer "in which city was she born" (relevant: no).

### 3.2 Agreement is not enough: Cohen's kappa

Two graders label the same items yes or no. The obvious measure is the **observed
agreement**: the share of items where they give the same label. Part 1 showed how it lies
when one label dominates.

**Cohen's kappa** removes the agreement two graders would reach by chance, given how often
each says yes. By hand, on the lists of the configuration cell (the human says yes 8 times
in 10, the judge 10 times in 10):

- observed agreement: they agree on the 8 items the human says yes to: `8 / 10 = 0.80`;
- agreement expected by chance: both say yes with probability `0.8 x 1.0`, both say no with
  `0.2 x 0.0`, total `0.80`;
- `kappa = (observed - expected) / (1 - expected) = (0.80 - 0.80) / 0.20 = 0`.

Kappa is 0: exactly what chance gives. 1 is perfect agreement; the usual reading puts
"substantial" above 0.6, which is the threshold of ADR 012.
[`cohen_kappa`](../../src/llm_testbench/eval/agreement.py#L38) returns the pieces as well as the number.

In [4]:
def show(name: str, human: list[bool], judge_labels: list[bool]) -> None:
    result = cohen_kappa(human, judge_labels)
    kappa = "undefined" if result.kappa is None else f"{result.kappa:.2f}"
    print(
        f"{name:<34} observed {result.observed:.2f}  expected by chance {result.expected:.2f}  "
        f"kappa {kappa}   (both yes {result.both_yes}, both no {result.both_no}, "
        f"human only {result.first_only}, judge only {result.second_only})"
    )


show("the lists of the configuration", TOY_HUMAN, TOY_JUDGE)
show("a judge that misses one bad answer", TOY_HUMAN, [*TOY_HUMAN[:-1], True])
show("a judge that agrees on everything", TOY_HUMAN, TOY_HUMAN)
lopsided = [True] * 48 + [False] * 2
show("48 yes in 50, one disagreement", lopsided, [True] * 49 + [False])
show("25 yes in 50, one disagreement", [True] * 25 + [False] * 25, [True] * 26 + [False] * 24)

the lists of the configuration     observed 0.80  expected by chance 0.80  kappa 0.00   (both yes 8, both no 0, human only 0, judge only 2)
a judge that misses one bad answer observed 0.90  expected by chance 0.74  kappa 0.62   (both yes 8, both no 1, human only 0, judge only 1)
a judge that agrees on everything  observed 1.00  expected by chance 0.68  kappa 1.00   (both yes 8, both no 2, human only 0, judge only 0)
48 yes in 50, one disagreement     observed 0.98  expected by chance 0.94  kappa 0.66   (both yes 48, both no 1, human only 0, judge only 1)
25 yes in 50, one disagreement     observed 0.98  expected by chance 0.50  kappa 0.96   (both yes 25, both no 24, human only 0, judge only 1)


The last two lines are the reason for the next section. With 48 "yes" in 50, **one**
disagreement takes kappa from 1.00 to 0.66: the number hangs on two items. With the labels
balanced, one disagreement costs 0.04. Kappa on a one-sided set is noise.

### 3.3 A calibration set with both verdicts in it

The items must be **real answers of the system**: a judge calibrated on invented examples
is calibrated for invented examples. But real answers on the gold context are nearly all
grounded and relevant, which is the one-sided set kappa cannot use.

[`build_items`](../../src/llm_testbench/eval/calibration.py#L72) draws 50 answers from the archived generation run, then
deliberately breaks 20 of them:

- 10 are shown against **another passage** of the same article: the answer loses its
  support (and stays on topic, the hard case);
- 10 are shown against **another question**: the answer stops being relevant.

Three details:

- an item ([`CalibrationItem`](../../src/llm_testbench/eval/calibration.py#L38)) stores **which example** its question, its
  passages and its answer come from, not the texts: [`materialize`](../../src/llm_testbench/eval/calibration.py#L137) reads them
  back from the dataset. The versioned file holds no dataset text;
- the swapped items are shuffled among the others, and the grader is never told which
  they are;
- **the swap is not the label.** A swapped passage may still support the answer by
  accident; a natural answer may be ungrounded (the 13 answers to unanswerable questions
  are natural items). The human labels all 50, blind, and those labels are the truth. The
  swaps only guarantee that "no" occurs.

The live cell produces four toy answers with the real generation runner (scripted
generator), draws the items, and makes one swapped item to show what the grader sees.

In [5]:
ANSWERS = [  # in the order of the fixture file: q-river, q-length, q-nobel, q-born
    "The Loire rises in the Massif Central [1].",
    "The Loire is 1,006 kilometres long [1].",
    "She received the Nobel Prize in Chemistry in 1911 [1].",
    "Marie Curie shared the Nobel Prize in Physics in 1903 [1].",
]
toy_run = await run_generation(
    squad,
    client=scripted("anthropic", *ANSWERS),
    builder=AnswerBuilder(prompts),
    model=GENERATOR,
    suite="notebook",
    arm="toy",
    sample_mode=True,
    resamples=200,
)
items = build_items(toy_run, squad, size=4, seed=0)
# Four answers are too few for the tool to swap any (it swaps at most a fifth each way):
# one swapped item is added by hand, the Loire answer against the Marie Curie excerpt.
river = next(item for item in items if item.answer_from == "q-river")
items.append(
    river.model_copy(
        update={"item_id": "c005", "passages_from": "q-nobel", "kind": "other_passage"}
    )
)
write_items(items, workdir)

print(f"{'item':<5} {'kind':<14} {'question of':<10} {'passages of':<12} answer")
for item in items:
    print(
        f"{item.item_id:<5} {item.kind:<14} {item.question_from:<10} {item.passages_from:<12} {item.answer}"
    )

material = materialize(items[-1], squad)
print(f"\nitem {items[-1].item_id} as a grader reads it:")
print(f"  question: {material.question}")
print(f"  excerpt : {material.evidence[0].text}")
print(f"  answer  : {material.answer}")

item  kind           question of passages of  answer
c001  natural        q-nobel    q-nobel      She received the Nobel Prize in Chemistry in 1911 [1].
c002  natural        q-river    q-river      The Loire rises in the Massif Central [1].
c003  natural        q-born     q-born       Marie Curie shared the Nobel Prize in Physics in 1903 [1].
c004  natural        q-length   q-length     The Loire is 1,006 kilometres long [1].
c005  other_passage  q-river    q-nobel      The Loire rises in the Massif Central [1].

item c005 as a grader reads it:
  question: Where does the Loire rise?
  excerpt : Marie Curie received the Nobel Prize in Physics in 1903, shared with Pierre Curie and Henri Becquerel. She received the Nobel Prize in Chemistry in 1911.
  answer  : The Loire rises in the Massif Central [1].


In [6]:
real_items = read_items(ROOT / CALIBRATION_DIR)
kinds = Counter(item.kind for item in real_items)
source = load_generation(
    ROOT / "data" / "results" / "runs" / "generation-baseline" / "squad-gold-sonnet.json"
)
answerable = {answer.example_id: answer.answerable for answer in source.scored()}
to_unanswerable = sum(1 for item in real_items if not answerable[item.answer_from])

print(f"the real calibration set: {len(real_items)} items drawn from {real_items[0].source_run}")
for kind, count in kinds.most_common():
    print(f"  {kind:<15} {count}")
print(f"answers given to questions that have no answer in their passage: {to_unanswerable}")

the real calibration set: 50 items drawn from 20261105T165120Z-generation-baseline-squad-gold-sonnet
  natural         30
  other_passage   10
  other_question  10
answers given to questions that have no answer in their passage: 13


The real set: 50 answers of `claude-sonnet-5-5` on SQuAD 2.0, 30 left as they are, 10 shown
against another passage, 10 against another question. **13 of the 50 are answers to
questions that have no answer in their passage**: the ungrounded-with-a-citation case of
notebook 1, the one this judge exists to catch, present in the set without any swap.

### 3.4 The human labels: blind, two questions, resumable

[`annotate`](../../src/llm_testbench/eval/annotate.py#L54) shows one item at a time, exactly what a judge would read and nothing
more: the question, the excerpt, the answer. Not whether the item was swapped; not what a
model thought of it. A label given knowing the expected answer measures obedience.

Two questions per item, a `y` or an `n` each: supported? relevant? The rules shown are the
rubrics of the judge in other words: the human and the judge must be answering the same
question, or their disagreement measures the wording. Each label is appended to
`labels.jsonl` as it is given: the session stops anywhere and resumes.

The live cell replays the labels of the configuration cell through the real tool. What is
printed is what the labeller sees (the first item only is shown in full).

In [7]:
keys: list[str] = []
for item in items:
    supported, relevant = (
        SWAPPED_LABEL if item.kind != "natural" else HUMAN_LABELS[item.answer_from]
    )
    keys += [supported, relevant, ""]  # the two answers, then an empty note
replay = iter(keys)
shown: list[str] = []
added = annotate(items, squad, workdir, read=lambda _prompt: next(replay), write=shown.append)

first_item_end = [i for i, line in enumerate(shown) if line.startswith("─")][1]
print("\n".join(shown[:first_item_end]))
print(f"[...]\n{shown[-1].strip()}")
labels = read_labels(workdir)
print(f"\n{'item':<5} {'kind':<14} supported  relevant")
for item in items:
    label = labels[item.item_id]
    print(f"{item.item_id:<5} {item.kind:<14} {label.groundedness!s:<10} {label.relevance}")

Two questions per item, answered from what is on screen and nothing else.

  Supported?  y = every claim of the answer is stated in the excerpts, or follows directly
              from them. n = at least one claim is not there, or is contradicted.
              What you know from elsewhere does not count. A citation marker such as [1]
              is not a proof: read the excerpt.
  Relevant?   y = the answer gives the kind of thing the question asks for, about the thing
              it asks about, even partially. n = it answers something else.
              Whether the answer is right, or supported, does not count here.

Keys: y, n — s skips the item, q stops (everything answered so far is saved), ? shows this.

0 of 5 items already labelled, 5 to go.

──────────────────────────────────────────────────────────────────────────────────────────────────
Item c001  (1 of 5)

QUESTION
  In what year did Marie Curie receive the Nobel Prize in Chemistry?

EXCERPT [1]
  Marie Curie received 

### 3.5 The calibration: each judge against the human

[`calibrate`](../../src/llm_testbench/eval/calibration.py#L199) has every judge grade every labelled item on both rubrics, then
computes, per judge and per rubric, the agreement with the human labels
([`JudgeAgreement`](../../src/llm_testbench/eval/calibration.py#L176)): observed agreement, kappa, the share of "yes" the judge
gives, the cost, and **the list of items where they differ**. That list is the useful
output: a disagreement is read before anything is concluded. Sometimes the judge is wrong;
sometimes the rubric is unclear; sometimes the human is.

Two judges are planned for the real run: `gpt-5.6-luna`, the cheap one that would grade
everything, and `gpt-5.6-terra`, ten times its price, on the 50 items only, to see what the
cheaper judge loses.

The live cell calibrates two scripted judges on the five toy items: one that says "yes" to
everything without reading, and one scripted to repeat the human. Five items prove nothing;
the point is to read the table.

In [8]:
always_yes = Judge(
    client=scripted("openai", verdict("yes", "Looks fine.")), prompts=prompts, model=JUDGE
)
# A judge scripted to give the human's own labels, in the order the calibration asks:
# for each item, groundedness then relevance.
echo_replies = []
for item in items:
    label = labels[item.item_id]
    echo_replies += [
        verdict("yes" if label.groundedness else "no", "Checked against the excerpt."),
        verdict("yes" if label.relevance else "no", "Compared with the question."),
    ]
careful = Judge(
    client=scripted("openai", *echo_replies),
    prompts=prompts,
    model=ModelRef.parse("openai/gpt-5.6-terra"),
)
report = await calibrate(items, labels, squad, [always_yes, careful])
print(format_calibration(report))

5 items labelled by hand (groundedness 60% yes, relevance 80% yes)
judge                        rubric        agreement  kappa  judge yes     cost  disagreements
openai/gpt-5.6-luna          groundedness        60%   0.00       100%   $0.002  c004, c005
openai/gpt-5.6-luna          relevance           80%   0.00       100%   $0.002  c003
openai/gpt-5.6-terra         groundedness       100%   1.00        60%   $0.002  none
openai/gpt-5.6-terra         relevance          100%   1.00        80%   $0.002  none


Read the first two lines. The judge that never reads **agrees with the human 60% of the
time on groundedness and 80% on relevance**, and its kappa is 0.00 on both. Agreement
reports how lopsided the set is; kappa reports the judge.

### 3.6 Three biases, probed with a control

A judge reads form as well as content. Three known weaknesses, each probed by changing one
thing that should not matter and grading again ([`probe`](../../src/llm_testbench/eval/bias.py#L93)):

| Bias | The change | A biased judge |
|---|---|---|
| **verbosity** | sentences that add no claim are appended to the answer ([`padded`](../../src/llm_testbench/eval/bias.py#L53)) | says "yes" more often to longer answers |
| **format** | the `[1]` markers are removed ([`without_citations`](../../src/llm_testbench/eval/bias.py#L57)) | treated the marker as proof, says "no" once it is gone |
| **position** | the excerpts are given in the opposite order ([`reversed_excerpts`](../../src/llm_testbench/eval/bias.py#L61)) | reads the first excerpt more carefully than the last |

The number is the share of items whose verdict **flips**. It means nothing alone: a judge
graded twice on the very same item flips sometimes too. So each item is also graded again
**unchanged** ([`repeat`](../../src/llm_testbench/eval/bias.py#L48), the control), and a bias is claimed only when a change
flips more verdicts than doing nothing, and in one direction. Flips that cancel out are
noise.

A fourth bias, **self-preference** (a judge kinder to answers written by its own family),
is handled by construction: the judge is OpenAI's, the generator Anthropic's. Measuring it
needs a second generator and a second judge; it is not paid for before the calibration
says the judge is usable at all.

The live cell probes a scripted judge that says "yes" until the citation marker is gone.

In [9]:
materials = [materialize(item, squad) for item in items if item.kind == "natural"]
# Order of the calls: both rubrics on the four items as they are (8), then, for each change
# and each rubric, the items again. The scripted judge says "no" on groundedness once the
# citation marker is gone, "yes" otherwise.
script = [verdict("yes", "Each claim is in the excerpt.")] * 8
for name in PERTURBATIONS_TO_RUN:
    for rubric in PERTURBATIONS[name][2]:
        blind = name == "without_citations" and rubric is Rubric.GROUNDEDNESS
        script += [
            verdict("no", "No citation, so no support.")
            if blind
            else verdict("yes", "Each claim is in the excerpt.")
        ] * len(materials)
biased = Judge(client=scripted("openai", *script), prompts=prompts, model=JUDGE)
print(format_bias(await probe(materials, biased, PERTURBATIONS_TO_RUN)))

judge                    change             probes     rubric        items flips   rate     95% interval  direction
openai/gpt-5.6-luna      repeat             control    groundedness      4     0   0.00     [0.00, 0.00]  0 to yes, 0 to no
openai/gpt-5.6-luna      repeat             control    relevance         4     0   0.00     [0.00, 0.00]  0 to yes, 0 to no
openai/gpt-5.6-luna      padded             verbosity  groundedness      4     0   0.00     [0.00, 0.00]  0 to yes, 0 to no
openai/gpt-5.6-luna      padded             verbosity  relevance         4     0   0.00     [0.00, 0.00]  0 to yes, 0 to no
openai/gpt-5.6-luna      without_citations  format     groundedness      4     4   1.00     [1.00, 1.00]  0 to yes, 4 to no
openai/gpt-5.6-luna      without_citations  format     relevance         4     0   0.00     [0.00, 0.00]  0 to yes, 0 to no
a change is a bias only if it flips more verdicts than the repeat control, one way


The control flips nothing, padding flips nothing, and removing the markers flips all four
groundedness verdicts towards "no": this scripted judge was reading the `[1]`, not the
excerpt. The rubric tells the real judge that "a marker is not a proof"; the probe checks
whether it listened.

## Part 4 — Run: where the real calibration stands

The archive cell reads the human labels and, when the calibration has been run
(`make eval-calibrate`), its report.

In [10]:
directory = ROOT / CALIBRATION_DIR
real_labels = read_labels(directory)
print(f"human labels: {len(real_labels)} of {len(real_items)} items")

report_path = ROOT / "data" / "results" / "calibration" / "judge_calibration.json"
if report_path.is_file():
    real_report = CalibrationReport.model_validate_json(report_path.read_text())
    print(format_calibration(real_report))
    print("\nacceptance rule (ADR 012): a judge is used on a rubric at kappa >= 0.6")
    for row in real_report.judges:
        kappa = row.agreement.kappa
        usable = kappa is not None and kappa >= 0.6
        print(
            f"  {row.judge:<24} {row.rubric.value:<13} "
            f"{'usable' if usable else 'NOT usable: revise the rubric, then try a stronger judge'}"
        )
elif real_labels:
    print("labels are there, the calibration has not been run: make eval-calibrate (about $0.40)")
else:
    print("the calibration waits for the labels: make eval-annotate (about one hour, resumable)")

human labels: 0 of 50 items
the calibration waits for the labels: make eval-annotate (about one hour, resumable)


Until the labels exist there is no judged number in this project, on purpose. The order is
fixed: labels, then `make eval-calibrate`, then the acceptance rule, then the judged
baseline (groundedness and relevance on the runs of notebook 1), then the bias probes
(`make eval-bias`).

## Part 5 — Pitfalls

**Agreement on a lopsided set is the share of the majority label.** 90% agreement on a set
that is 90% "yes" is what a judge that never reads gets. Report kappa, and report the
balance of the set next to it.

**The swapped items are the easy ones.** An answer about the Loire against an excerpt about
Marie Curie is caught by any judge. The hard cases are the natural ones: an answer that
cites a real excerpt and adds one claim the excerpt does not make. The kappa measured here
is an upper bound on what the judge does on subtle failures; the list of disagreements, on
the 13 answers to unanswerable questions in particular, says more than the number.

**One labeller.** There is no measure of how consistent the human is with himself, and the
"truth" is one person's reading of two rules. Kappa on 50 items also has an uncertainty of
its own, about ±0.2: a judge at 0.65 and a judge at 0.55 are not distinguishable.

**The human and the judge must be asked the same question.** If the labeller counts "true
in the real world" as supported and the rubric does not, their disagreement measures the
rules, not the judge. The rules shown by the annotation tool are the rubric, reworded.

**A judge cut by its output cap fails loudly, which is the good case.** The judge reasons
before it writes, like the generator; its first output cap here was 400 tokens, sized for
a two-sentence justification. A cut reply is not valid JSON, the structured-output layer
asks for a repair, and the cost doubles before anything is graded. The cap is now 2,000.
The same mistake on the generator was silent (notebook 1, part 5).

**A verdict belongs to a rubric version.** Change one sentence of the prompt and the judge
is another judge: every verdict carries the version and the fingerprint of the rubric it
was given under ([`Verdict`](../../src/llm_testbench/eval/judge.py#L56)), and a calibration holds for that version only.

**The judge is not deterministic either.** That is what the repeat control of the bias
probes measures. A flip rate of 4% between two prompts means nothing if grading the same
item twice flips 5%.

**A judge that passes its calibration is calibrated for these answers.** SQuAD paragraphs,
one excerpt, short answers. Five chunks of a scientific paper are another reading task;
QASPER and HotpotQA answers join the calibration set if the judge is used on them.

## Part 6 — Guided experiments

1. **Agreement against kappa** (part 3.2). Set `TOY_JUDGE = [False] + [True] * 9`: the
   judge now says no once, to an answer the human accepts, and still yes to the two the
   human rejects. Agreement is 0.70, which sounds like a judge; kappa is -0.15, worse than
   chance. Then make the human's list half "yes", half "no"
   (`TOY_HUMAN = [True] * 5 + [False] * 5`) with a judge that gets one wrong
   (`TOY_JUDGE = [True] * 6 + [False] * 4`): agreement 0.90, kappa 0.80.
2. **Your labels** (parts 3.4 and 3.5). Change a label in `HUMAN_LABELS`, for instance
   `"q-born": ("y", "y")` (you decide the prize sentence is relevant enough). The
   always-yes judge now agrees with you 100% on relevance, and its kappa becomes
   *undefined*: with a single label on both sides there is nothing to measure. That is the
   one-sided set, in five items.
3. **The probes** (part 3.6). Add `"reversed_excerpts"` to `PERTURBATIONS_TO_RUN`: it
   produces no line, because every toy item has a single excerpt and a single excerpt
   cannot be reordered. A probe reports on the items it could change, and says how many.

For the real thing: `make eval-annotate`, then `make eval-calibrate`, then rerun this
notebook; part 4 fills in.

## Part 7 — The numbers

| | |
|---|---|
| Calibration items | 50 answers of `claude-sonnet-5-5` on SQuAD 2.0 (run of 2026-11-05) |
| Natural / other passage / other question | 30 / 10 / 10 |
| Answers to questions without an answer, among the 50 | 13 |
| Human labels | **0 on 2026-11-05**: to be given |
| Judges to calibrate | `openai/gpt-5.6-luna` (would grade everything), `openai/gpt-5.6-terra` (on the 50 items only) |
| Acceptance rule | kappa at least 0.6 per rubric, written in ADR 012 before any label |
| Estimated cost of the calibration | about $0.40 |

No published score applies: a calibration is between one judge, one rubric and one
labeller. For orientation only, the literature reports agreement between LLM judges and
people that varies widely with the task and the rubric; this project quotes none of those
numbers as a target and measures its own.

**The third-party evaluators.** Ragas, DeepEval and promptfoo all ship judge-based metrics.
Ragas' *faithfulness* splits the answer into claims and checks each against the context,
which is the method of our groundedness rubric, in two model calls instead of one; its
*answer relevancy* generates questions from the answer and compares them with the real
one, a different idea from our relevance rubric. What none of them ships is the
calibration: they provide the judge, not the measurement of the judge on your answers.
Running Ragas' faithfulness on these 50 items, next to our judge and the human labels, is a
few cents and one dependency in the `bench` group; it was proposed with the plan of the
phase and waits for a decision.

## Part 8 — Interview questions

<details>
<summary>Six questions, with the answers expected (click to open)</summary>

**1. You use an LLM to grade your RAG's answers. How do you know the grader is right?**
By measuring its agreement with a person on a sample of real answers, with a measure that
discounts chance (Cohen's kappa), on a set where both verdicts occur. And by reading the
disagreements. Without that the judge's scores are unverified model outputs.

**2. Your judge agrees with human labels 92% of the time. Is it good?**
Unknown. If 92% of the answers are good, a judge that says "good" to everything scores
92%. Ask for kappa and for the share of "yes" in the labels. In this notebook a judge that
never reads agrees 60% and 80% of the time with a kappa of zero.

**3. Why a yes/no verdict rather than a score from 1 to 10?**
A scale needs graders who mean the same thing by each level, and enough items per level to
check it. Two graders do not share a "7". A binary verdict on a narrow question ("is every
claim in the excerpts?") can be calibrated with 50 items; several narrow yes/no rubrics
say more than one wide score.

**4. Why ask for the justification before the verdict?**
A model writes left to right. Verdict first, it commits and then produces the
justification that fits; justification first, the verdict is conditioned on the reasoning.
The order is enforced by the output schema, not by a request in the prompt.

**5. Name three biases of an LLM judge and how you would detect each.**
Verbosity (longer reads as better): pad answers with content-free sentences, count the
verdicts that flip. Format (a citation marker reads as proof): strip the markers, same
count. Self-preference (kinder to its own family): cross two generators with two judges.
Always against a control: grade the same items twice unchanged, because a judge flips some
verdicts by itself.

**6. Groundedness and relevance: give an answer that passes one and fails the other.**
"Marie Curie shared the Nobel Prize in Physics in 1903" to the question "where was she
born", with an excerpt about her prizes: grounded, not relevant. The reverse: "She was born
in Warsaw" from the same excerpt: relevant, true, and not grounded, because the excerpt
does not say it.

</details>

## Part 9 — What we did not do, and why

- **The calibration itself.** It needs 50 labels given by hand: Christophe's, about an hour.
  Everything in this notebook that is a real number about the judge waits for them.
- **A rubric for correctness** (does the answer agree with the reference answer?). QASPER
  needs it: "contains the reference answer" is 0.28 on the gold evidence there. It comes
  with the judged baseline, and is calibrated like the other two.
- **The self-preference probe**: a second generator, a second judge, about a dollar. Not
  before the first calibration says the judge is usable.
- **Ragas, DeepEval, promptfoo** on the same items: proposed, awaiting a decision (part 7).
- **A second labeller.** One person, so no agreement between people, which is the ceiling
  any judge should be compared with.
- **A scale, or partial credit.** Yes or no, per rubric. An answer that is half supported
  is not supported.

## Glossary

| Term | Meaning |
|---|---|
| **judge (LLM-as-judge)** | a model that grades an answer by following a rubric |
| **rubric** | the grader's instructions: what to check, what to ignore, how to conclude |
| **groundedness** | every claim of the answer is stated in the excerpts the system was given |
| **relevance** | the answer addresses the question that was asked |
| **verdict** | the judge's conclusion on one rubric: yes or no, with its justification |
| **calibration** | measuring a judge's agreement with human labels before using it |
| **calibration set** | the items labelled by hand that the judge is measured on |
| **observed agreement** | share of items on which two graders give the same label |
| **Cohen's kappa** | agreement beyond what chance would give: 0 is chance, 1 is perfect |
| **blind labelling** | the labeller sees what the judge sees, and nothing about what is expected |
| **bias (of a judge)** | a sensitivity to something that is not what the rubric asks |
| **verbosity bias** | longer answers graded more kindly |
| **format bias** | the form of an answer (markers, lists) graded instead of its content |
| **position bias** | what comes first graded differently from what comes last |
| **self-preference** | a judge kinder to answers written by a model of its own family |
| **repeat control** | grading the same items twice unchanged, to know the judge's own noise |
| **structured output** | a reply constrained to a schema (here: a justification, then yes or no) |